**Problem Statement:**

Develop a system that detects multiple types of toxic comments (multi-label
classification), identifies linguistic patterns associated with each toxicity type, and
analyzes relationships between toxicity categories.

## Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download("stopwords")
nltk.download("wordnet")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer
from mlxtend.preprocessing import TransactionEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer

from mlxtend.frequent_patterns import fpgrowth, association_rules
from sklearn.preprocessing import MultiLabelBinarizer

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Import Datasets

In [ ]:
# Datasets
test_df = pd.read_csv("/content/test.csv")
test_labels_df = pd.read_csv("/content/test_labels.csv")
train_df = pd.read_csv("/content/train.csv")

In [ ]:
test_df.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [ ]:
test_labels_df.head(8)

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,-1,-1,-1,-1,-1,-1
1,0000247867823ef7,-1,-1,-1,-1,-1,-1
2,00013b17ad220c46,-1,-1,-1,-1,-1,-1
3,00017563c3f7919a,-1,-1,-1,-1,-1,-1
4,00017695ad8997eb,-1,-1,-1,-1,-1,-1
5,0001ea8717f6de06,0,0,0,0,0,0
6,00024115d4cbde0f,-1,-1,-1,-1,-1,-1
7,000247e83dcc1211,0,0,0,0,0,0


In [ ]:
train_df.head(10)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
5,00025465d4725e87,"""\n\nCongratulations from me as well, use the ...",0,0,0,0,0,0
6,0002bcb3da6cb337,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,1,1,0,1,0
7,00031b1e95af7921,Your vandalism to the Matt Shirvington article...,0,0,0,0,0,0
8,00037261f536c51d,Sorry if the word 'nonsense' was offensive to ...,0,0,0,0,0,0
9,00040093b2687caa,alignment on this subject and which are contra...,0,0,0,0,0,0


## Label Distribution

In [ ]:
# Define toxicity label columns (multi-label classification targets)
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print(train_df[label_cols].sum())

toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64


## Basic Text Analysis

In [ ]:
# Create comment length feature
train_df["comment_length"] = train_df["comment_text"].apply(len)

# Summary statistics
print(train_df["comment_length"].describe())

count    159571.000000
mean        394.073221
std         590.720282
min           6.000000
25%          96.000000
50%         205.000000
75%         435.000000
max        5000.000000
Name: comment_length, dtype: float64


## Text Preprocessing

In [ ]:
def clean_text(text):
  text = str(text).lower()
  text = re.sub(r'[^a-z\s]', "", text)  # remove punctuation
  text = re.sub(r'\s+', " ", text).strip()  # remove extra spaces
  text = re.sub(r'http\S+', "", text)  # remove URLs
  text = re.sub(r'<.*?>', "", text)  # remove HTML
  return text

In [ ]:
# Stopwords and lemmatizer for text processing
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Improved text cleaning function
def improved_clean_text(text):
  text = text.lower()  # lowercase
  text = re.sub(r'[^a-z\s]', "", text)  # remove punctuation and numbers
  text = re.sub(r'\s+', " ", text).strip()  # remove extra spaces
  text = re.sub(r'http\S+', "", text)  # remove URLs
  text = re.sub(r'<.*?>', "", text)  # remove HTML tags
  text = re.sub(r'(.)\1+', r'\1\1', text)  # reduce repeated characters

  # Tokenize text into words
  words = text.split()

  # Remove stopwords
  words = [w for w in words if w not in stop_words]

  # Lemmatize words
  words = [lemmatizer.lemmatize(w) for w in words]

  # Return cleaned text
  return " ".join(words)

In [ ]:
# Apply text cleaning to training and test datasets
train_df["clean_text"] = train_df["comment_text"].apply(improved_clean_text)
test_df["clean_text"] = test_df["comment_text"].apply(improved_clean_text)

## Features and Labels

In [ ]:
# Split dataset into X(features) and y(targets)
X = train_df["clean_text"]
y = train_df[label_cols]

### Train and Validation Split

In [ ]:
# Train-test split for model evaluation
X_train, X_val, y_train, y_val =train_test_split(X, y, test_size= 0.2, random_state=42)

## TF-IDF Vectorization

In [ ]:
# Convert text into numerical features (capture word importance)
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

# Fit TF-IDF on training data and transform validation and test
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test_df["clean_text"])

# Models

## Base Model: Logistic Regression

In [ ]:
# OnevsRest classifer to train one classifier per label
model = OneVsRestClassifier(LogisticRegression())
model.fit(X_train_tfidf, y_train)

OneVsRestClassifier(estimator=LogisticRegression())

### Model Evaluation

In [ ]:
# Predict on validation set and evaluate
y_pred = model.predict(X_val_tfidf)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.62      0.74      3056
           1       0.59      0.26      0.37       321
           2       0.91      0.63      0.74      1715
           3       0.47      0.11      0.18        74
           4       0.82      0.52      0.64      1614
           5       0.72      0.17      0.28       294

   micro avg       0.87      0.56      0.68      7074
   macro avg       0.74      0.39      0.49      7074
weighted avg       0.86      0.56      0.67      7074
 samples avg       0.06      0.05      0.05      7074



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Model 2: Support Vector Machine

In [ ]:
# Linear SVM classifier
svm_model =OneVsRestClassifier(LinearSVC(max_iter=2000))
svm_model.fit(X_train_tfidf, y_train)

OneVsRestClassifier(estimator=LinearSVC(max_iter=2000))

### Predictions

In [ ]:
# Predict on val data
y_pred_svm = svm_model.predict(X_val_tfidf)

### Evaluation

In [ ]:
#  Evaluate
print("SVM Results:")
print(classification_report(y_val, y_pred_svm, target_names=label_cols))

SVM Results:
               precision    recall  f1-score   support

        toxic       0.87      0.68      0.76      3056
 severe_toxic       0.50      0.24      0.32       321
      obscene       0.88      0.69      0.77      1715
       threat       0.54      0.19      0.28        74
       insult       0.80      0.56      0.66      1614
identity_hate       0.69      0.27      0.39       294

    micro avg       0.84      0.61      0.71      7074
    macro avg       0.71      0.44      0.53      7074
 weighted avg       0.83      0.61      0.70      7074
  samples avg       0.06      0.05      0.06      7074



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Model 3: LSTM

In [ ]:
# Build vocabulary from training text
def build_vocab(texts, max_vocab_size=20000):
  counter = Counter()

  # Count word frequency
  for text in texts:
    counter.update(text.split())

  # Assign index to most common words
  vocab = {word:i+1 for i, (word, _) in enumerate(counter.most_common(max_vocab_size))}
  vocab["<PAD>"] = 0
  return vocab

vocab = build_vocab(train_df["clean_text"])

### Sequence Conversion

In [ ]:
# Convert text to fixed-length integer sequences
def text_to_sequence(text, vocab, max_len=200):
    # Map word to indices
    seq = [vocab.get(word, 0) for word in text.split()]

    # Truncate long texts
    seq = seq[:max_len]

    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))

    return seq

# Convert all text into sequences
X_seq = [text_to_sequence(text, vocab) for text in train_df["clean_text"]]

In [ ]:
# Convert labels to numpy array
y =train_df[label_cols].values

In [ ]:
# Train-Validation split
X_train_seq, X_val_seq, y_train, y_val = train_test_split(X_seq, y, test_size=0.2, random_state=42)

### Dataset Class

In [ ]:
class ToxicDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)  # Input sequences
        self.y = torch.tensor(y, dtype=torch.float32)  # Label output

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets and dataloaders
train_dataset = ToxicDataset(X_train_seq, y_train)
val_dataset = ToxicDataset(X_val_seq, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

### Model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, output_dim=6):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        x = hidden[-1]
        x = self.fc(x)
        return self.sigmoid(x)

# Initialize model
model = LSTMModel(len(vocab))

In [ ]:
# Loss function and optimizer
criterion = nn.BCELoss()  # Binary cross entropy for multi-label
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training function
def train_model(model, loader):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        outputs = model(X_batch) # Forward pass
        loss = criterion(outputs, y_batch)  # Compute loss

        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# Evaluation function
def evaluate_model(model, loader):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            preds.append(outputs.numpy())
            true.append(y_batch.numpy())

    return np.vstack(preds), np.vstack(true)

In [ ]:
# Train for 3 epochs
for epoch in range(3):
    loss = train_model(model, train_loader)
    print(f"Epoch {epoch+1}, Loss: {loss}")

Epoch 1, Loss: 0.1454399122704838
Epoch 2, Loss: 0.14024223136834632
Epoch 3, Loss: 0.09979700158375845


In [ ]:
# Evaluate
preds, true = evaluate_model(model, val_loader)

# Convert probabilities to binary predictions
preds_binary = (preds > 0.5).astype(int)

# Classification report
print(classification_report(true, preds_binary, target_names=label_cols))

               precision    recall  f1-score   support

        toxic       0.82      0.66      0.73      3056
 severe_toxic       0.62      0.05      0.09       321
      obscene       0.82      0.72      0.76      1715
       threat       0.00      0.00      0.00        74
       insult       0.70      0.63      0.67      1614
identity_hate       0.00      0.00      0.00       294

    micro avg       0.79      0.60      0.68      7074
    macro avg       0.49      0.34      0.38      7074
 weighted avg       0.74      0.60      0.66      7074
  samples avg       0.06      0.05      0.05      7074



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Pattern Mining

## Frequency Analysis

In [ ]:
# Combine all cleaned comments into single list of words
all_words = " ".join(train_df["clean_text"]).split()

# Count frequency of each word
word_counts = Counter(all_words)

# Display 20 most common words
print(word_counts.most_common(20))

[('article', 72967), ('page', 56491), ('wikipedia', 35602), ('talk', 31814), ('one', 29936), ('please', 29616), ('would', 29212), ('like', 28131), ('dont', 26102), ('see', 21699), ('source', 21455), ('think', 20727), ('also', 20549), ('know', 20007), ('im', 19476), ('time', 18763), ('people', 18747), ('edit', 17594), ('use', 16323), ('make', 16176)]


Result: Most common words are stopwords with  clean_text() function.

In [ ]:
# Extract words from toxic and non-toxic comments
toxic_words = " ".join(train_df[train_df['toxic']==1]['clean_text']).split()
non_toxic_words = " ".join(train_df[train_df['toxic']==0]['clean_text']).split()

# Count word frequencies for each
toxic_counts = Counter(toxic_words)
non_toxic_counts = Counter(non_toxic_words)

# Display most frequent words
print("Top toxic words:", toxic_counts.most_common(20))
print("Top non-toxic words:", non_toxic_counts.most_common(20))

Top toxic words: [('fuck', 8688), ('suck', 4477), ('shit', 3603), ('like', 3603), ('dont', 3566), ('nigger', 3417), ('wikipedia', 3263), ('fucking', 3194), ('u', 3071), ('go', 2972), ('hate', 2665), ('faggot', 2633), ('as', 2605), ('page', 2553), ('get', 2431), ('know', 2277), ('gay', 2249), ('die', 2077), ('im', 2052), ('fat', 1967)]
Top non-toxic words: [('article', 71098), ('page', 53938), ('wikipedia', 32339), ('talk', 30538), ('please', 28708), ('one', 28375), ('would', 28203), ('like', 24528), ('dont', 22536), ('source', 20949), ('see', 20770), ('also', 19818), ('think', 19406), ('know', 17730), ('time', 17515), ('im', 17424), ('people', 16821), ('edit', 16681), ('use', 15881), ('may', 15274)]


## N-Gram Analysis (Phrase Patterns)

In [ ]:
# Filter dataset into toxic/non-toxic subsets
toxic_text = train_df[train_df["toxic"] == 1]["clean_text"]
non_toxic_text = train_df[train_df["toxic"] == 0]["clean_text"]

### Toxic Bigrams

In [ ]:
# Extract top bigrams from toxic comments (pairs of words)
vectorizer = CountVectorizer(ngram_range=(2,2), max_features=20)

X_toxic = vectorizer.fit_transform(toxic_text)

# Get frequency counts of bigrams
counts = X_toxic.sum(axis=0).A1
words = vectorizer.get_feature_names_out()

# Sort by frequency
toxic_bigrams = sorted(zip(words, counts), key=lambda x: x[1], reverse=True)

# Remove repetitive bigrams
toxic_filtered_bigrams = [bg for bg in toxic_bigrams if bg[0].split()[0] != bg[0].split()[1]]

print("Top Toxic Bigrams:")
print(toxic_bigrams)
print(toxic_filtered_bigrams)

Top Toxic Bigrams:
[('nigger nigger', np.int64(2019)), ('fuck fuck', np.int64(1817)), ('moron hi', np.int64(1474)), ('hi moron', np.int64(1472)), ('faggot faggot', np.int64(1378)), ('pig pig', np.int64(1251)), ('jew fat', np.int64(1234)), ('fat jew', np.int64(1226)), ('shit shit', np.int64(1149)), ('go fuck', np.int64(1145)), ('as as', np.int64(1127)), ('hate hate', np.int64(1124)), ('bark bark', np.int64(999)), ('wanker wanker', np.int64(963)), ('suck suck', np.int64(882)), ('fuck go', np.int64(859)), ('ball ball', np.int64(833)), ('bullshit bullshit', np.int64(833)), ('nipple nipple', np.int64(763)), ('talk page', np.int64(733))]
[('moron hi', np.int64(1474)), ('hi moron', np.int64(1472)), ('jew fat', np.int64(1234)), ('fat jew', np.int64(1226)), ('go fuck', np.int64(1145)), ('fuck go', np.int64(859)), ('talk page', np.int64(733))]


### Non-Toxic Bigrams

In [ ]:
#  Extract bigrams from non-toxic comments
X_clean = vectorizer.fit_transform(non_toxic_text)

# Get frequency counts of bigrams
counts = X_clean.sum(axis=0).A1
words = vectorizer.get_feature_names_out()

# Sort by frequency
clean_bigrams = sorted(zip(words, counts), key=lambda x: x[1], reverse=True)

# Remove repetitive bigrams
clean_filtered_bigrams = [bg for bg in clean_bigrams if bg[0].split()[0] != bg[0].split()[1]]

print("Top Non-Toxic Bigrams:")
print(clean_bigrams)
print(clean_filtered_bigrams)

Top Non-Toxic Bigrams:
[('talk page', np.int64(14679)), ('speedy deletion', np.int64(4388)), ('would like', np.int64(3530)), ('reliable source', np.int64(3001)), ('dont know', np.int64(2715)), ('fair use', np.int64(2713)), ('personal attack', np.int64(2488)), ('feel free', np.int64(2321)), ('blocked editing', np.int64(2304)), ('dont think', np.int64(2144)), ('please stop', np.int64(2089)), ('im sure', np.int64(2008)), ('talk contribs', np.int64(1860)), ('edit summary', np.int64(1756)), ('welcome wikipedia', np.int64(1737)), ('article talk', np.int64(1699)), ('please see', np.int64(1689)), ('ip address', np.int64(1664)), ('user page', np.int64(1654)), ('let know', np.int64(1624))]
[('talk page', np.int64(14679)), ('speedy deletion', np.int64(4388)), ('would like', np.int64(3530)), ('reliable source', np.int64(3001)), ('dont know', np.int64(2715)), ('fair use', np.int64(2713)), ('personal attack', np.int64(2488)), ('feel free', np.int64(2321)), ('blocked editing', np.int64(2304)), ('dont

## Association Rule Mining

### FP Growth

In [ ]:
# Sample a subset of data
df_sample = train_df.sample(2000, random_state=42)

# Convert text into unigram features
vectorizer = CountVectorizer(
    ngram_range=(1,1),
    max_features=60,
    min_df=5,
    stop_words='english'
)

X = vectorizer.fit_transform(df_sample["clean_text"])
words = vectorizer.get_feature_names_out()

# Build transactions combining words and toxicity labels
transactions = []

for i in range(X.shape[0]):
    row = X[i]
    indices = row.indices

    items = [words[j] for j in indices]

    labels = [
        label for label in label_cols
        if df_sample.iloc[i][label] == 1
    ]

    # Keep meaningful transactions (word + label)
    if len(items) > 0 and len(labels) > 0:
        transactions.append(items + labels)

# Limit dataset size
transactions = transactions[:1500]

# Convert transactions into one-hot encoded format
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_onehot = pd.DataFrame(te_array, columns=te.columns_)

# Generate frequent itemsets
frequent_itemsets = fpgrowth(
    df_onehot,
    min_support=0.008,
    use_colnames=True
)

# Generate association rules based on confidence
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.25
)

# Only keep rules predicting labels
rules = rules[
    rules["consequents"].apply(lambda x: any(label in x for label in label_cols))
]

# Filter out weak relationships
rules = rules[rules["lift"] > 0.9]

# Sort rules by confidence
rules = rules.sort_values(by="confidence", ascending=False)

print(rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(10))

                                antecedents  \
5462819                              (hate)   
5462814                    (right, editing)   
5462810              (right, editing, page)   
5462807                    (right, editing)   
45         (wikipedia, insult, page, toxic)   
44        (wikipedia, obscene, page, toxic)   
42       (wikipedia, obscene, insult, page)   
5449775                      (added, toxic)   
5449774                    (obscene, added)   
5449769                      (thanks, time)   

                                  consequents   support  confidence       lift  
5462819                               (toxic)  0.020690         1.0   1.043165  
5462814                         (page, toxic)  0.013793         1.0   5.000000  
5462810                               (toxic)  0.013793         1.0   1.043165  
5462807                               (toxic)  0.013793         1.0   1.043165  
45                                  (obscene)  0.020690         1.0   1.98630

### Improved FP-Growth

In [ ]:
# Sample data
df_sample = train_df.sample(2000, random_state=42)

# Extend stopword list with specific terms
stop_words = set(stopwords.words('english'))

custom_words = {
    "wikipedia", "page", "editing", "article",
    "talk", "edit", "source", "use", "one",
    "would", "like", "get"
}
stop_words = list(stop_words.union(custom_words))

# Re-vectorize using refined stopword list
vectorizer = CountVectorizer(
    ngram_range=(1,1),
    max_features=60,
    min_df=5,
    stop_words=stop_words
)

X = vectorizer.fit_transform(df_sample["clean_text"])
words = vectorizer.get_feature_names_out()

# Build transactions
transactions = []

for i in range(X.shape[0]):
    row = X[i]
    indices = row.indices

    items = [words[j] for j in indices]

    labels = [
        label for label in label_cols
        if df_sample.iloc[i][label] == 1
    ]

    if len(items) > 0 and len(labels) > 0:
        transactions.append(items + labels)

# Limit size
transactions = transactions[:1500]

# One-hot encoding
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_onehot = pd.DataFrame(te_array, columns=te.columns_)

# Generate frequent itemsets
frequent_itemsets = fpgrowth(
    df_onehot,
    min_support=0.008,
    use_colnames=True
)

# Generate rules
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.25
)

# Only keep rules predicting labels
rules = rules[
    rules["consequents"].apply(lambda x: any(label in x for label in label_cols))
]

# Filter weak rules
rules = rules[rules["lift"] > 0.9]

# Sort rules by confidence
rules = rules.sort_values(by="confidence", ascending=False)

print(rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(10))

                       antecedents                   consequents   support  \
13            (insult, dont, stop)                       (toxic)  0.028777   
135519              (insult, hate)                     (obscene)  0.014388   
135520             (hate, obscene)                      (insult)  0.014388   
135522     (insult, hate, obscene)                       (toxic)  0.014388   
135523       (insult, hate, toxic)                     (obscene)  0.014388   
135524      (hate, obscene, toxic)                      (insult)  0.014388   
135525              (insult, hate)              (obscene, toxic)  0.014388   
135526             (hate, obscene)               (insult, toxic)  0.014388   
117787  (think, even, time, thing)  (stop, obscene, good, toxic)  0.014388   
117789  (think, even, stop, toxic)  (obscene, time, good, thing)  0.014388   

        confidence       lift  
13             1.0   1.045113  
135519         1.0   1.957746  
135520         1.0   2.074627  
135522       